# GolStats - Silver Events

This notebook transforms the raw event data from the Bronze layer
into a clean and analytics-ready Silver dataset.

The main objectives are:

- Flatten nested StatsBomb structures.
- Standardize column names.
- Extract common event attributes.
- Preserve the event-level granularity.
- Prepare the data for analytical transformations in the Gold layer.

Flow:
![image_1788520854513.png](./image_1788520854513.png "image_1788520854513.png")

## 1. Read Bronze data

The Bronze layer contains the original StatsBomb event structure,
including nested objects and arrays.

In this step, we load the Bronze Delta table as the source
for the Silver transformation.

In [0]:
# Read the Bronze event table as the source for the Silver transformation.

df_bronze = spark.table("golstats.bronze.eventos_statsbomb")

display(df_bronze)

In [0]:
df_bronze.printSchema()

## 2. Select and standardize common event attributes

StatsBomb stores many attributes inside nested structures.

In this step, we extract the most relevant fields and standardize
their names to create a consistent event-level dataset.

The goal is to keep the information that is useful across different
event types while preserving the original event granularity.

In [0]:
from pyspark.sql.functions import col

In [0]:
df_silver = (
    df_bronze
    .select(
        col("id").alias("event_id"),
        col("match_id"),
        col("index"),
        col("timestamp"),
        col("period"),
        col("minute"),
        col("second"),

        col("type.name").alias("event_type"),

        col("team.id").alias("team_id"),
        col("team.name").alias("team"),

        col("player.id").alias("player_id"),
        col("player.name").alias("player"),

        col("position.name").alias("position"),

        col("possession"),
        col("possession_team.id").alias("possession_team_id"),
        col("possession_team.name").alias("possession_team"),

        col("location").getItem(0).alias("x"),
        col("location").getItem(1).alias("y"),

        col("play_pattern.name").alias("play_pattern"),

        col("under_pressure"),
        col("counterpress"),

        col("source_file")
    )
)

display(df_silver)

In [0]:
df_silver.printSchema()

In [0]:
display(
    df_silver
    .select(
        "event_id",
        "match_id",
        "event_type",
        "team",
        "player",
        "position",
        "x",
        "y",
        "minute",
        "second"
    )
    .limit(20)
)

## 3. Extract event-specific attributes

Some StatsBomb attributes are only available for specific event types.

For example, pass information is stored inside the `pass` structure,
while shot information is stored inside the `shot` structure.

In this step, we extract the most relevant attributes for football
analysis while keeping all events at the event level.

Fields that do not apply to a particular event will remain null.

In [0]:
from pyspark.sql.functions import col

df_silver = (
    df_bronze
    .select(
        # Event identification
        col("id").alias("event_id"),
        col("match_id"),
        col("index"),
        col("timestamp"),

        # Time information
        col("period"),
        col("minute"),
        col("second"),

        # Event classification
        col("type.name").alias("event_type"),

        # Team information
        col("team.id").alias("team_id"),
        col("team.name").alias("team"),

        # Player information
        col("player.id").alias("player_id"),
        col("player.name").alias("player"),
        col("position.name").alias("position"),

        # Possession information
        col("possession"),
        col("possession_team.id").alias("possession_team_id"),
        col("possession_team.name").alias("possession_team"),

        # Event location
        col("location").getItem(0).alias("x"),
        col("location").getItem(1).alias("y"),

        # General event context
        col("play_pattern.name").alias("play_pattern"),
        col("under_pressure"),
        col("counterpress"),

        # Pass attributes
        col("pass.length").alias("pass_length"),
        col("pass.angle").alias("pass_angle"),
        col("pass.outcome.name").alias("pass_outcome"),
        col("pass.recipient.id").alias("pass_recipient_id"),
        col("pass.recipient.name").alias("pass_recipient"),
        col("pass.technique.name").alias("pass_technique"),
        col("pass.height.name").alias("pass_height"),
        col("pass.cross").alias("pass_cross"),
        col("pass.cut_back").alias("pass_cut_back"),
        col("pass.through_ball").alias("pass_through_ball"),
        col("pass.switch").alias("pass_switch"),
        col("pass.goal_assist").alias("pass_goal_assist"),

        # Pass destination
        col("pass.end_location").getItem(0).alias("pass_end_x"),
        col("pass.end_location").getItem(1).alias("pass_end_y"),

        # Shot attributes
        col("shot.statsbomb_xg").alias("shot_xg"),
        col("shot.outcome.name").alias("shot_outcome"),
        col("shot.technique.name").alias("shot_technique"),
        col("shot.body_part.name").alias("shot_body_part"),
        col("shot.first_time").alias("shot_first_time"),
        col("shot.one_on_one").alias("shot_one_on_one"),

        # Shot destination
        col("shot.end_location").getItem(0).alias("shot_end_x"),
        col("shot.end_location").getItem(1).alias("shot_end_y")
    )
)

display(df_silver)

In [0]:
display(
    df_silver
    .filter(col("event_type") == "Pass")
    .select(
        "event_id",
        "team",
        "player",
        "x",
        "y",
        "pass_end_x",
        "pass_end_y",
        "pass_length",
        "pass_outcome",
        "pass_recipient"
    )
    .limit(20)
)

In [0]:
display(
    df_silver
    .filter(col("event_type") == "Shot")
    .select(
        "event_id",
        "match_id",
        "team",
        "player",
        "x",
        "y",
        "shot_xg",
        "shot_outcome",
        "shot_technique",
        "shot_body_part"
    )
    .limit(20)
)

## 4. Normalize pass outcomes

StatsBomb does not explicitly store an outcome for every completed pass.

When the `pass.outcome` field is null, the pass is considered completed.

To make the Silver dataset easier to analyze, we normalize these values
into an explicit `pass_outcome` classification:

- `null` → `Complete`
- Existing outcomes are preserved.

In [0]:
from pyspark.sql.functions import col, when

df_silver = (
    df_silver
    .withColumn(
        "pass_outcome",
        when(
            (col("event_type") == "Pass") & col("pass_outcome").isNull(),
            "Complete"
        ).otherwise(col("pass_outcome"))
    )
)

display(
    df_silver
    .filter(col("event_type") == "Pass")
    .select(
        "event_id",
        "team",
        "player",
        "pass_outcome",
        "pass_recipient",
        "pass_length"
    )
    .limit(20)
)

In [0]:
display(
    df_silver
    .filter(col("event_type") == "Pass")
    .groupBy("pass_outcome")
    .count()
    .orderBy("count", ascending=False)
)

## 5. Investigate pass outcome consistency

During the previous validation, we observed that the `pass_outcome` field contains
multiple values, including `Complete`, `Incomplete`, `Out`, `Pass Offside`,
`Injury Clearance`, and `Unknown`.

We also observed that some passes may contain a `pass_recipient` even when the
`pass_outcome` is not `Complete`.

Before applying any additional transformation, we will investigate the relationship
between these two fields.

The objective is to determine whether `pass_recipient` can be used as a reliable
indicator of a completed pass, or whether `pass_outcome` should remain the primary
source for classifying pass completion.

This validation will help us avoid introducing assumptions into the Silver layer
and ensure that the resulting dataset preserves the semantics of the original
StatsBomb data.
 

In [0]:
from pyspark.sql.functions import col, count

display(
    df_silver
    .filter(col("event_type") == "Pass")
    .groupBy("pass_outcome")
    .agg(
        count("*").alias("total_passes"),
        count("pass_recipient").alias("passes_with_recipient")
    )
    .orderBy("total_passes", ascending=False)
)

### Investigation findings

The validation shows that `pass_recipient` cannot be used as a reliable indicator
of pass completion.

All `Complete` passes in the current dataset have a recipient, but a significant
number of `Incomplete` passes also contain a recipient.

Therefore, `pass_recipient` represents the intended or recorded recipient of the
pass and should not be used to override `pass_outcome`.

For the Silver transformation, `pass_outcome` will remain the primary field for
pass completion classification.

Null `pass.outcome` values are normalized to `Complete`, while existing outcome
values such as `Incomplete`, `Out`, `Pass Offside`, and `Injury Clearance` are
preserved.

The `pass_recipient` field will be retained because it can support future
player-to-player passing and passing network analysis in the Gold layer.


## 6. Silver data quality validation

Before writing the Silver dataset to Delta, we perform a set of basic data quality checks.

The objective is to verify that:

* The total number of events remains unchanged from Bronze.
* All expected matches are still present.
* Event identifiers remain unique.
* Critical fields such as `event_id`, `match_id`, and `event_type` do not contain unexpected null values.

These checks help ensure that the Silver transformation changed the structure
of the data without unintentionally losing or duplicating events.


In [0]:
from pyspark.sql.functions import col, count, countDistinct

display(
    df_silver.agg(
        count("*").alias("total_events"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("event_id").alias("unique_event_ids")
    )
)

## 7. Validate critical fields

The next validation checks the completeness of the key identification and classification fields.

The following columns are considered critical for the Silver dataset:

* `event_id`
* `match_id`
* `event_type`

These fields are required to identify each event, associate it with a match,
and determine the type of event.

Unexpected null values in these columns could affect downstream analytics,
so they are validated before persisting the Silver table.


In [0]:
display(
    df_silver.select(
        count("*").alias("total_rows"),
        count("event_id").alias("events_with_id"),
        count("match_id").alias("events_with_match_id"),
        count("event_type").alias("events_with_type")
    )
)

## 8. Validate spatial data

StatsBomb events can contain spatial information such as the starting location
of an event and the destination of a pass or shot.

These fields are not expected to be populated for every event type.

For example, events such as `Starting XI` or `Substitution` may not contain
a location, while passes and shots should normally contain spatial coordinates.

This validation checks the availability of spatial data by event type and helps
identify unexpected missing values without treating valid NULL values as data
quality issues.


In [0]:
display(
    df_silver
    .groupBy("event_type")
    .agg(
        count("*").alias("total_events"),
        count("x").alias("events_with_location"),
        count("pass_end_x").alias("passes_with_destination"),
        count("shot_end_x").alias("shots_with_destination")
    )
    .orderBy("total_events", ascending=False)
)

## 9. Validate event-specific attributes

The Silver dataset contains attributes that are specific to certain event types.

For example:

* Pass events contain passing attributes such as `pass_length`, `pass_outcome`,
  and `pass_recipient`.
* Shot events contain shooting attributes such as `shot_xg`, `shot_outcome`,
  and `shot_technique`.

These attributes are expected to be populated only when the corresponding
event type applies.

This validation checks that event-specific fields are correctly associated
with their respective event types before the Silver dataset is persisted.


In [0]:
display(
    df_silver
    .filter(col("event_type") != "Pass")
    .agg(
        count("pass_length").alias("non_pass_events_with_pass_length"),
        count("pass_outcome").alias("non_pass_events_with_pass_outcome"),
        count("pass_recipient").alias("non_pass_events_with_recipient"),
        count("pass_end_x").alias("non_pass_events_with_pass_destination")
    )
)

## 10. Validate shot-specific attributes

Shot-specific attributes should only be populated for `Shot` events.

This validation checks that shooting attributes such as expected goals (`shot_xg`),
shot outcome, technique, and shot destination are not populated for unrelated
event types.

This ensures that the Silver transformation correctly preserves the relationship
between event types and their specific attributes.


In [0]:
display(
    df_silver
    .filter(col("event_type") != "Shot")
    .agg(
        count("shot_xg").alias("non_shot_events_with_xg"),
        count("shot_outcome").alias("non_shot_events_with_outcome"),
        count("shot_technique").alias("non_shot_events_with_technique"),
        count("shot_end_x").alias("non_shot_events_with_shot_destination")
    )
)

## 11. Silver validation summary

The Silver transformation has passed the main structural and consistency checks.

The validation confirmed that:

* All 35,142 Bronze events were preserved.
* All 10 matches are present.
* Event identifiers remain unique.
* Critical identification fields contain no unexpected null values.
* Spatial information is correctly preserved for Pass and Shot events.
* Pass-specific attributes are only populated for Pass events.
* Shot-specific attributes are only populated for Shot events.
* `pass_recipient` is not used as a proxy for pass completion.
* Null pass outcomes are normalized to `Complete`, while existing outcomes are preserved.

The Silver dataset is now ready to be persisted as a Delta table.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS golstats.silver;

In [0]:
SILVER_TABLE = "golstats.silver.eventos"

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
%sql
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT match_id) AS total_matches,
    COUNT(DISTINCT event_id) AS unique_event_ids
FROM golstats.silver.eventos;